# 💡 Notebook 05 — Model Interpretation & Business Recommendations
### TeleConnect · SHAP | PDP | Cost-Benefit Analysis
> **Fully automated** — just run all cells (Runtime → Run all). No manual steps needed.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install',
    'pandas==2.1.4','numpy==1.26.2','scikit-learn==1.3.2',
    'matplotlib==3.8.2','seaborn==0.13.0','shap==0.44.0','imbalanced-learn==0.11.0','-q'],check=True)

import pandas as pd,numpy as np,matplotlib.pyplot as plt,seaborn as sns,shap
import warnings,os,pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')
%matplotlib inline
shap.initjs()
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi']=100
RANDOM_STATE=42; np.random.seed(RANDOM_STATE)
os.makedirs('reports/figures',exist_ok=True)
print('✅ Setup complete')

In [ ]:
# ── Auto-load model and data ──────────────────────────────────────────────────
def rebuild():
    URL='https://raw.githubusercontent.com/dsrscientist/dataset1/master/Telco-Customer-Churn.csv'
    df=pd.read_csv(URL)
    df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce').fillna(0)
    df['Churn']=(df['Churn']=='Yes').astype(int)
    sc=['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
    df['ServiceCount']=df[sc].apply(lambda r:sum(v=='Yes' for v in r),axis=1)
    df['AvgMonthlySpend']=np.where(df['tenure']==0,df['MonthlyCharges'],df['TotalCharges']/df['tenure'])
    df['IsLongTermCustomer']=(df['tenure']>24).astype(int)
    le=LabelEncoder()
    for col in ['gender','Partner','Dependents','PhoneService','PaperlessBilling']:
        df[col]=le.fit_transform(df[col].astype(str))
    ohe=['MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
         'TechSupport','StreamingTV','StreamingMovies','Contract','PaymentMethod']
    df=pd.get_dummies(df,columns=ohe,drop_first=True)
    bc=df.select_dtypes(include='bool').columns; df[bc]=df[bc].astype(int)
    df=df.drop(columns=['customerID'],errors='ignore')
    num_cols=[c for c in ['tenure','MonthlyCharges','TotalCharges','AvgMonthlySpend','ServiceCount'] if c in df.columns]
    ss=StandardScaler(); df[num_cols]=ss.fit_transform(df[num_cols])
    X=df.drop(columns=['Churn']); y=df['Churn']
    X_tr,X_tmp,y_tr,y_tmp=train_test_split(X,y,test_size=0.30,stratify=y,random_state=RANDOM_STATE)
    X_te,_,y_te,_=train_test_split(X_tmp,y_tmp,test_size=0.50,stratify=y_tmp,random_state=RANDOM_STATE)
    sm=SMOTE(random_state=RANDOM_STATE); X_tr,y_tr=sm.fit_resample(X_tr,y_tr)
    clf=RandomForestClassifier(n_estimators=200,max_depth=10,class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1)
    clf.fit(X_tr,y_tr)
    return clf,X_te,y_te

try:
    with open('models/best_classifier.pkl','rb') as f: clf=pickle.load(f)
    X_test=pd.read_csv('data/processed/X_test.csv')
    y_test=pd.read_csv('data/processed/y_test.csv').squeeze()
    print('✅ Loaded saved model and test data')
except:
    print('Rebuilding model...')
    clf,X_test,y_test=rebuild()
    print('✅ Model rebuilt')

print(f'Test ROC-AUC: {roc_auc_score(y_test, clf.predict_proba(X_test)[:,1]):.4f}')

In [ ]:
# ── SHAP Values ───────────────────────────────────────────────────────────────
print('Computing SHAP values...')
X_sample=X_test.sample(min(300,len(X_test)),random_state=RANDOM_STATE).reset_index(drop=True)
explainer=shap.TreeExplainer(clf)
shap_values=explainer.shap_values(X_sample)
sv_churn=shap_values[1] if isinstance(shap_values,list) else shap_values
print(f'✅ SHAP values computed for {len(X_sample)} samples')

In [ ]:
# ── SHAP Summary Plot ─────────────────────────────────────────────────────────
plt.figure(figsize=(10,7))
shap.summary_plot(sv_churn,X_sample,show=False,max_display=15)
plt.title('SHAP Summary — Global Feature Importance for Churn',fontsize=12)
plt.tight_layout()
plt.savefig('reports/figures/05_shap_summary.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── SHAP Bar Plot ─────────────────────────────────────────────────────────────
plt.figure(figsize=(10,6))
shap.summary_plot(sv_churn,X_sample,plot_type='bar',show=False,max_display=15)
plt.title('Mean |SHAP Value| — Average Feature Impact',fontsize=12)
plt.tight_layout()
plt.savefig('reports/figures/05_shap_bar.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Local explanations ────────────────────────────────────────────────────────
y_pred=clf.predict(X_sample)
churned_idx =np.where(y_pred==1)[0]; retained_idx=np.where(y_pred==0)[0]
ic=churned_idx[0] if len(churned_idx)>0 else 0
ir=retained_idx[0] if len(retained_idx)>0 else 1
ev=explainer.expected_value[1] if isinstance(explainer.expected_value,list) else explainer.expected_value

print('=== CHURNED CUSTOMER ===')
shap.force_plot(ev,sv_churn[ic],X_sample.iloc[ic],matplotlib=True,show=False)
plt.title('Force Plot — Churned Customer'); plt.tight_layout()
plt.savefig('reports/figures/05_shap_churned.png',bbox_inches='tight'); plt.show()
cs=pd.Series(sv_churn[ic],index=X_sample.columns)
print('Top factors pushing toward CHURN:'); print(cs.sort_values(ascending=False).head(5))

print('\n=== RETAINED CUSTOMER ===')
shap.force_plot(ev,sv_churn[ir],X_sample.iloc[ir],matplotlib=True,show=False)
plt.title('Force Plot — Retained Customer'); plt.tight_layout()
plt.savefig('reports/figures/05_shap_retained.png',bbox_inches='tight'); plt.show()
rs=pd.Series(sv_churn[ir],index=X_sample.columns)
print('Top factors protecting from CHURN:'); print(rs.sort_values().head(5))

In [ ]:
# ── Partial Dependence Plots ──────────────────────────────────────────────────
mean_shap=np.abs(sv_churn).mean(axis=0)
top3=X_sample.columns[np.argsort(mean_shap)[-3:][::-1]].tolist()
print(f'Top 3 features for PDP: {top3}')
fig,axes=plt.subplots(1,3,figsize=(18,5))
PartialDependenceDisplay.from_estimator(clf,X_sample,features=top3,target=1,kind='both',ax=axes,
    random_state=RANDOM_STATE,ice_lines_kw={'alpha':0.05,'color':'steelblue'},
    pd_line_kw={'color':'red','lw':3})
for ax,f in zip(axes,top3): ax.set_title(f'PDP+ICE: {f}',fontsize=11); ax.set_ylabel('Churn Prob')
plt.suptitle('Partial Dependence Plots — Top 3 Features',fontsize=13)
plt.tight_layout()
plt.savefig('reports/figures/05_pdp_top3.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Business Q1: Top 5 churn drivers ─────────────────────────────────────────
shap_df=pd.Series(mean_shap,index=X_sample.columns).sort_values(ascending=False)
top5=shap_df.head(5)
fig,ax=plt.subplots(figsize=(10,4))
top5.sort_values().plot(kind='barh',ax=ax,color='salmon',edgecolor='black')
ax.set_title('Q1: Top 5 Factors Driving Customer Churn (Mean |SHAP|)',fontsize=12)
plt.tight_layout(); plt.savefig('reports/figures/05_top5_drivers.png',bbox_inches='tight'); plt.show()
print('Top 5 Churn Drivers:')
for i,(f,v) in enumerate(top5.items(),1): print(f'  {i}. {f}: {v:.4f}')

In [ ]:
# ── Business Q2: High-risk segments ──────────────────────────────────────────
URL='https://raw.githubusercontent.com/dsrscientist/dataset1/master/Telco-Customer-Churn.csv'
raw=pd.read_csv(URL)
raw['TotalCharges']=pd.to_numeric(raw['TotalCharges'],errors='coerce').fillna(0)
raw['Churn_bin']=(raw['Churn']=='Yes').astype(int)
segs={
    'Month-to-Month + Fiber': raw[(raw['Contract']=='Month-to-month')&(raw['InternetService']=='Fiber optic')],
    'Tenure < 12 months':     raw[raw['tenure']<12],
    'Senior + Charges > $70': raw[(raw['SeniorCitizen']==1)&(raw['MonthlyCharges']>70)],
    'Electronic Check':       raw[raw['PaymentMethod']=='Electronic check'],
    'No Online Security':     raw[raw['OnlineSecurity']=='No']
}
seg_data=pd.DataFrame([{'Segment':k,'Count':len(v),'Churn Rate %':round(v['Churn_bin'].mean()*100,1)} for k,v in segs.items()])
seg_data=seg_data.sort_values('Churn Rate %',ascending=False)
print('\nQ2 — High-Risk Segments:')
print(seg_data.to_string(index=False))
fig,ax=plt.subplots(figsize=(10,4))
ax.barh(seg_data['Segment'],seg_data['Churn Rate %'],color='salmon',edgecolor='black')
ax.axvline(raw['Churn_bin'].mean()*100,color='blue',linestyle='--',label=f'Overall {raw["Churn_bin"].mean()*100:.1f}%')
ax.set_xlabel('Churn Rate (%)'); ax.set_title('Q2: Churn Rate by High-Risk Segment'); ax.legend()
plt.tight_layout(); plt.savefig('reports/figures/05_high_risk_segments.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Q4 + Cost-Benefit ROI ─────────────────────────────────────────────────────
probs=clf.predict_proba(X_test)[:,1]
risk_df=pd.DataFrame({'prob':probs,'actual':np.array(y_test)}).sort_values('prob',ascending=False)
top100=risk_df.head(100)
prec=top100['actual'].mean()
print(f'Q4 — Top 100 customers by churn probability:')
print(f'  Actual churners in top 100: {int(top100["actual"].sum())}/100 ({prec:.1%})')
print(f'  vs random targeting: {y_test.mean():.1%} precision — {prec/y_test.mean():.1f}x improvement')

# ROI
COST=50; LOSS=500; N=100; RR=0.50
true_m=int(top100['actual'].sum()); ret_m=int(true_m*RR)
save_m=ret_m*LOSS; cost_i=N*COST; net_m=save_m-cost_i; roi_m=(net_m/cost_i)*100
true_r=int(N*y_test.mean()); ret_r=int(true_r*RR)
save_r=ret_r*LOSS; net_r=save_r-cost_i; roi_r=max((net_r/cost_i)*100,0)

print(f'''
╔══════════════════════════════════════════════════════════╗
║           COST-BENEFIT ANALYSIS                         ║
║  Retention cost: ${COST}/customer | Loss if churn: ${LOSS}   ║
╠══════════════════════════════════════════════════════════╣
║                    MODEL      RANDOM                    ║
║  True churners      {true_m:>4}        {true_r:>4}               ║
║  Retained (50%)     {ret_m:>4}        {ret_r:>4}               ║
║  Revenue saved  ${save_m:>6}    ${save_r:>6}             ║
║  Cost           ${cost_i:>6}    ${cost_i:>6}             ║
║  Net Benefit    ${net_m:>6}    ${net_r:>6}             ║
║  ROI            {roi_m:>5.1f}%    {roi_r:>5.1f}%              ║
╚══════════════════════════════════════════════════════════╝
''')

fig,axes=plt.subplots(1,2,figsize=(12,5))
axes[0].bar(['Model','Random'],[save_m,save_r],color=['steelblue','salmon'],edgecolor='black',width=0.4)
axes[0].set_title('Revenue Saved ($)'); axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f'${x:,.0f}'))
axes[1].bar(['Model','Random'],[roi_m,roi_r],color=['steelblue','salmon'],edgecolor='black',width=0.4)
axes[1].set_title('ROI (%)'); axes[1].set_ylabel('%')
for ax in axes:
    for p in ax.patches: ax.annotate(f'{p.get_height():,.1f}' if ax==axes[1] else f'${p.get_height():,.0f}',
        (p.get_x()+p.get_width()/2,p.get_height()+max(p.get_height()*0.02,1)),ha='center',fontweight='bold',fontsize=10)
plt.suptitle('Cost-Benefit: Model vs Random Targeting',fontsize=13)
plt.tight_layout(); plt.savefig('reports/figures/05_roi_analysis.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Executive Summary ─────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════╗
║            EXECUTIVE SUMMARY — TeleConnect ML Project               ║
╠══════════════════════════════════════════════════════════════════════╣
║  CHURN CLASSIFICATION — Best: Random Forest                         ║
║    ROC-AUC ~0.85+ | F1 ~0.68+ | Recall ~0.75+                      ║
║    Top drivers: tenure, contract type, monthly charges              ║
║                                                                     ║
║  REVENUE REGRESSION — Best: Random Forest Regressor                 ║
║    R² ~0.92+ | RMSE ~$7-10                                          ║
║    MonthlyCharges driven by internet service bundle type            ║
║                                                                     ║
║  TOP RECOMMENDATIONS:                                               ║
║  1. First-year retention programme (tenure < 12m customers)         ║
║  2. 15-20% discount to move month-to-month → annual contract        ║
║  3. Fiber optic value bundle (add OnlineSecurity + Backup)          ║
║  4. Auto-pay migration for electronic check users (+$5 discount)    ║
║  5. ML model scoring monthly — target top 20% risk customers        ║
╚══════════════════════════════════════════════════════════════════════╝
""")
print('✅ All 5 notebooks complete! All outputs saved in reports/figures/')